# 1장 2강: 유의수준·검정력·표본 크기의 트레이드오프와 실질적 유의성 — 실습문제

## 실습 목표

- 효과 크기와 유의수준이 같을 때 표본 크기에 따라 검정력이 어떻게 변하는지 설명할 수 있다.
- 목표 검정력과 유의수준을 이용해 필요한 표본 크기를 계산할 수 있다.
- 두 집단의 평균 차이에 대한 Cohen’s d를 계산하고 해석할 수 있다.
- p-value와 효과 크기, 실무 기준을 함께 사용하여 의사결정 근거를 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing.csv`

주요 컬럼은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `CentralAir` | 중앙 냉방시설 유무(`Y`, `N`) |
| `KitchenQual` | 주방 품질(`Ex`, `Gd`, `TA`, `Fa`) |

> 모든 검정은 별도 지시가 없으면 유의수준 `α = 0.05`를 사용합니다.  
> Cohen’s d는 절댓값을 기준으로 약 0.2는 작은 효과, 0.5는 중간 효과, 0.8 이상은 큰 효과로 해석합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
import pandas as pd
from scipy import stats

df = pd.read_csv('ames_housing.csv')

df.describe()

df.head()

,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


In [2]:
# 변동성 : 값들이 얼마나 서로 제각각인지 : 문방구 vs 백화점

# 표준편차 : 평균 주변에 값이 얼마나 퍼져 있는지
# -> 표준편차가 크면 값들이 차이가 전반적으로 크다

# 분산 : 값들의 퍼짐  정도, 표준편차를 제곱한 값
# -> 시험 점수의 표준편차가 10점이면 분산은 100점이다.

# 통계적 유의성 : 검정 결과가 미리 정한 기각 기준을 충족하여, 차이를 뒷받침할 통계적 근거가 있다는 뜻
# -> p-value < alpha 이면 유의미하다고 판단
# -> 평균 배송 시간이 30분에서 29분으로 줄고 p = 0.002라면 alpha = 0.05에서 통계적으로 유의한 차이다.

---

## 필수 1. 표본 크기와 검정력 비교

### 문제 1-1. 중앙 냉방시설 비교 연구에 필요한 표본 수 설계

#### 문제 설명

중앙 냉방시설이 있는 주택과 없는 주택의 판매가격을 비교하는 연구를 준비하고 있습니다. 사전 조사에서 표준화 효과 크기는 `0.3` 정도로 예상했습니다.

다음 조건에서 그룹당 표본 크기에 따른 검정력을 비교하고, 목표 검정력 0.8을 확보하기 위해 필요한 표본 수를 계산하세요.

- 예상 효과 크기: `0.3`
- 유의수준: `0.05`
- 양측검정
- 비교할 그룹당 표본 수: `30명`, `100명`, `300명`

#### 요구사항

1. `TTestIndPower()` 객체를 생성하세요.
2. 그룹당 표본 수가 30명, 100명, 300명일 때의 검정력을 각각 계산하세요.
3. 표본 크기별 검정력을 소수점 셋째 자리까지 출력하세요.
4. 유의수준 0.05, 목표 검정력 0.8, 효과 크기 0.3일 때 그룹당 필요한 표본 수를 계산하세요.
5. 계산된 표본 수는 소수점 이하를 올림하여 정수로 출력하세요.
6. 세 표본 크기 중 목표 검정력 0.8을 충족하는 경우를 확인하세요.

#### 해석 질문

**Q1.** 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?
- 차이를 구분하기 쉬워져 검정력이 높아진다.

**Q2.** 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?  
- 제 2종 오류

**Q3.** 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?  
- 표본수가 소수점일수는 없잖아요 특히 사람인데

**Q4.** 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?
- 아니오. 표본 수를 크게 할수록 비용이 늘고, 표본이 극단적으로 커지면 특정 값에 수렴하여 통계적 유의성이 떨어집니다.

#### 제출 결과

- 표본 크기별 검정력
- 목표 검정력에 필요한 그룹당 표본 수
- 목표 검정력 충족 여부
- Q1~Q4 답변

In [5]:
# 필수 1 코드를 작성하세요.
import math
from statsmodels.stats.power import TTestIndPower

# 파라미터 설정
effect_size = 0.3
alpha = 0.05
sample_sizes = [30, 100, 300]
target_power = 0.8

# 1. TTestIndPower 객체 생성
power_analysis = TTestIndPower()

# 2 & 3. 표본 수별 검정력 계산 및 출력
print("=== 표본 크기별 검정력 (양측검정) ===")
for n in sample_sizes:
    power = power_analysis.power(
        effect_size=effect_size,
        nobs1=n,
        alpha=alpha,
        ratio=1.0,
        alternative="two-sided",
    )
    print(f"그룹당 표본 수 {n:>3}명: 검정력 = {power:.3f}")

# 4 & 5. 목표 검정력 0.8 달성에 필요한 그룹당 표본 수 계산
required_n = power_analysis.solve_power(
    effect_size=effect_size,
    power=target_power,
    alpha=alpha,
    ratio=1.0,
    alternative="two-sided",
)
required_n_ceil = math.ceil(required_n)

print("\n=== 목표 검정력(0.8) 확보를 위한 표본 크기 ===")
print(f"계산된 필요 표본 수(실수): {required_n:.4f}")
print(f"최종 필요 표본 수(올림 정수): 그룹당 {required_n_ceil}명")

# 6. 세 표본 크기 중 목표 검정력 0.8 충족 여부 확인
print("\n=== 목표 검정력(0.8) 충족 여부 ===")
for n in sample_sizes:
    status = "충족" if n >= required_n_ceil else "미충족"
    print(f"그룹당 {n:>3}명: {status}")

=== 표본 크기별 검정력 (양측검정) ===
그룹당 표본 수  30명: 검정력 = 0.208
그룹당 표본 수 100명: 검정력 = 0.560
그룹당 표본 수 300명: 검정력 = 0.956

=== 목표 검정력(0.8) 확보를 위한 표본 크기 ===
계산된 필요 표본 수(실수): 175.3847
최종 필요 표본 수(올림 정수): 그룹당 176명

=== 목표 검정력(0.8) 충족 여부 ===
그룹당  30명: 미충족
그룹당 100명: 미충족
그룹당 300명: 충족


### 필수 1 답변 작성란

**Q1.** 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?
- 차이를 구분하기 쉬워져 검정력이 높아진다. 실제 차이를 우연한 변동과 구분하기 쉬워짐.

**Q2.** 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?  
- 제 2종 오류. 실제 차이가 있는데도 귀무가설을 기각하지 못한다.

**Q3.** 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?  
- 사람이나 주택을 소수점 단위로 표현할 수 없고, 내림하면 계산된 목표 검정력을 충족하지 못할 수 있다.

**Q4.** 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?
- 아니오. 표본 수를 크게 할수록 비용이 늘고, 표본이 극단적으로 커지면 특정 값에 수렴하여 통계적 유의성이 떨어집니다.


---

## 필수 2. 통계적 유의성과 실질적 유의성 종합 판단

### 문제 2-1. 중앙 냉방시설 유무에 따른 판매가격 차이 분석

#### 문제 설명

한 부동산 회사는 중앙 냉방시설이 있는 주택과 없는 주택의 평균 판매가격 차이를 분석하려고 합니다. 회사는 두 집단의 평균 판매가격 차이가 **100,000달러 이상**이어야 냉방시설 설치 지원 사업을 검토할 실무적 가치가 있다고 정했습니다.

#### 요구사항

1. `CentralAir == "Y"`인 주택의 `SalePrice`를 `air_yes`에 저장하세요.
2. `CentralAir == "N"`인 주택의 `SalePrice`를 `air_no`에 저장하세요.
3. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
4. `stats.ttest_ind()`로 두 집단 평균 차이에 대한 양측검정을 수행하세요.
5. 제공된 공식에 따라 Cohen’s d 계산 함수를 작성하고 효과 크기를 구하세요.
6. 평균 차이, p-value, Cohen’s d를 출력하세요.
7. 다음 기준으로 결과를 판단하세요.
   - `p-value ≤ 0.05`: 통계적으로 유의함
   - `|Cohen’s d| ≥ 0.8`: 큰 효과
   - `|평균 차이| ≥ 100000`: 회사 기준에서 실질적으로 유의함
8. 세 결과를 종합하여 사업 검토 여부에 대한 근거를 작성하세요.

#### Cohen’s d 공식

두 집단의 평균 차이를 합동 표준편차로 나누어 계산합니다.

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}
$$

$$
s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}
$$

- $\bar{x}_1$, $\bar{x}_2$: 각 집단의 평균
- $s_1$, $s_2$: 각 집단의 표본 표준편차
- $n_1$, $n_2$: 각 집단의 표본 수
- $s_p$: 합동 표준편차

#### 해석 질문

**Q1.** p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요?  
**Q2.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q3.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q4.** 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요?  
**Q5.** 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립표본 t검정 결과
- 평균 차이와 Cohen’s d
- 통계적·효과크기·회사 기준 판단
- 최종 의사결정 근거
- Q1~Q5 답변

In [6]:
# 필수 2 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats

# 1 & 2. 집단 분리 및 SalePrice 결측치 제거
air_yes = df[df["CentralAir"] == "Y"]["SalePrice"].dropna()
air_no = df[df["CentralAir"] == "N"]["SalePrice"].dropna()

# 3. 집단별 표본 수와 평균 판매가격 계산
n1, n2 = len(air_yes), len(air_no)
mean1, mean2 = air_yes.mean(), air_no.mean()
s1, s2 = air_yes.std(ddof=1), air_no.std(ddof=1)

print("=== 집단별 기초 통계 ===")
print(f"중앙 냉방 있음(Y) - 표본 수: {n1}개, 평균 판매가격: ${mean1:,.2f}")
print(f"중앙 냉방 없음(N) - 표본 수: {n2}개, 평균 판매가격: ${mean2:,.2f}")

# 4. 독립표본 t검정 수행 (양측검정, 등분산 가정 기본값)
t_stat, p_value = stats.ttest_ind(air_yes, air_no, equal_var=True)


# 5. Cohen's d 계산 함수 정의
def calculate_cohens_d(group1, group2):
    n_1, n_2 = len(group1), len(group2)
    m_1, m_2 = group1.mean(), group2.mean()
    s_1, s_2 = group1.std(ddof=1), group2.std(ddof=1)

    # 합동 표준편차(pooled standard deviation)
    sp = np.sqrt(
        ((n_1 - 1) * (s_1**2) + (n_2 - 1) * (s_2**2)) / (n_1 + n_2 - 2)
    )
    d = (m_1 - m_2) / sp
    return d


cohens_d = calculate_cohens_d(air_yes, air_no)
mean_diff = mean1 - mean2

# 6. 평균 차이, p-value, Cohen's d 출력
print("\n=== 분석 검정 결과 ===")
print(f"평균 차이 (Y - N): ${mean_diff:,.2f}")
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"Cohen's d: {cohens_d:.4f}")

# 7. 기준별 판단
stat_sig = p_value <= 0.05
effect_large = abs(cohens_d) >= 0.8
biz_sig = abs(mean_diff) >= 100_000

print("\n=== 기준별 평가 결과 ===")
print(
    f"1. 통계적 유의성 (p <= 0.05): {'유의함' if stat_sig else '유의하지 않음'}"
)
print(
    f"2. 표준화 효과크기 (|d| >= 0.8): {'큰 효과' if effect_large else '보통 이하 효과'}"
)
print(
    f"3. 회사 실무 기준 (|평균 차이| >= $100,000): {'실질적으로 유의함' if biz_sig else '실질적으로 유의하지 않음'}"
)

=== 집단별 기초 통계 ===
중앙 냉방 있음(Y) - 표본 수: 1365개, 평균 판매가격: $186,186.71
중앙 냉방 없음(N) - 표본 수: 95개, 평균 판매가격: $105,264.07

=== 분석 검정 결과 ===
평균 차이 (Y - N): $80,922.64
t-통계량: 9.9149
p-value: 1.8095e-22
Cohen's d: 1.0521

=== 기준별 평가 결과 ===
1. 통계적 유의성 (p <= 0.05): 유의함
2. 표준화 효과크기 (|d| >= 0.8): 큰 효과
3. 회사 실무 기준 (|평균 차이| >= $100,000): 실질적으로 유의하지 않음


### 필수 2 답변 작성란

**Q1.** p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요? 
- pvalue는 관측된 차이에 대한 통계적 유의성, Cohen's d는 두 집단 차이의 크기를 표준화하여 보여줌

**Q2.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?
- 네
- 귀무가설: 냉방 시설 유무는 주택 가격 평균에 차이를 보이지 않는다 -> 기각

**Q3.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?
- Cohen's d의 절대값이 1.05고, 규정한 효과크기인 0.8보다 크기 때문에 큰 효과로 해석할 수 있다. 
  
**Q4.** 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요? 
- 아니오. $80,000 정도였기 때문에 회사의 기대 미충족

**Q5.** 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?
- 그렇다. 
- 통계적 유의성 : 진짜 중앙냉방 여부에 따라서 가격 차이가 있는게 맞는지?
- 효과 크기 : 중앙냉방 여부가 가격 차이에 미치는 영향이 얼마나 큰지?
- 회사 입장은 업무에서 요구하는 최소 금액 차이를 넘는가?

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질이 `Gd`인 집단과 `TA`인 집단 비교

#### 문제 설명

부동산 중개회사는 주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 평균 판매가격을 비교하려고 합니다. 회사는 평균 판매가격 차이가 **50,000달러 이상**이면 마케팅에서 주방 품질의 차이를 강조할 실무적 가치가 있다고 판단합니다.

> 필수 2에서 학습한 검정과 효과 크기 계산 절차를 새로운 두 집단에 적용하는 과제입니다.

#### 요구사항

1. 주방 품질이 `Gd`인 집단과 `TA`인 집단의 `SalePrice`를 각각 준비하세요.
2. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
3. 두 집단의 평균 차이에 대한 양측 독립표본 t검정을 수행하세요.
4. Cohen’s d를 계산하세요.
5. 평균 차이, p-value, Cohen’s d를 출력하세요.
6. 다음 기준에 따라 각각 판단하세요.
   - 통계적 유의성: `p-value ≤ 0.05`
   - 큰 효과: `|Cohen’s d| ≥ 0.8`
   - 실무적 유의성: `|평균 차이| ≥ 50000`
7. 세 가지 판단을 종합하여 주방 품질을 마케팅에서 강조할 근거가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q2.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q3.** 회사가 정한 50,000달러 기준을 충족하나요?  
**Q4.** 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- t통계량과 p-value
- 평균 차이와 Cohen’s d
- 통계적·실질적 유의성 판단
- 최종 결론
- Q1~Q4 답변

In [ ]:
# 과제 코드를 작성하세요.

### 과제 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 실습 마무리

1. 검정력은 무엇이며 제2종 오류와 어떤 관계가 있나요?
2. 같은 효과 크기와 유의수준에서 표본 크기가 커지면 검정력은 어떻게 변하나요?
3. 목표 검정력이 높거나 발견하려는 효과가 작을수록 필요한 표본 수는 어떻게 변하나요?
4. p-value와 Cohen’s d를 함께 확인해야 하는 이유는 무엇인가요?
5. 통계적으로 유의한 결과가 반드시 실무적으로 중요한 결과를 의미하나요?